# Równanie normalne

**Plan notatnika:**
1. Idea równania normalnego
2. Dane
3. Macierz cech z kolumną jedynek
4. Obliczenia krok po kroku
5. Porównanie z `LinearRegression`
6. Podsumowanie

## 1. Idea równania normalnego

**Równanie normalne** pozwala policzyć parametry regresji liniowej **bezpośrednio ze wzoru**, bez iteracji:

$$\mathbf{w} = (X^T X)^{-1} X^T \mathbf{y}$$

- $\mathbf{w} = [w_0, w_1, \ldots]^T$ – wektor parametrów modelu,
- $X$ – macierz cech z dodatkową kolumną jedynek (dla wyrazu wolnego $w_0$),
- $\mathbf{y}$ – wektor wartości docelowych.

W zapisie macierzowym model to $\hat{\mathbf{y}} = X\mathbf{w}$. Wzór wynika z rozwiązania zadania „znajdź $\mathbf{w}$ minimalizujące MSE”.

**Kiedy działa dobrze:** mało cech i danych.
**Ograniczenia:**
- odwracanie macierzy jest kosztowne dla wielu cech (koszt rośnie około sześciennie z liczbą cech),
- macierz $X^T X$ nie da się odwrócić, gdy cechy są współliniowe.

## 2. Dane

Zależność wynagrodzenia od lat pracy (6 osób).

In [1]:
import numpy as np

lata_pracy = np.array([1, 2, 3, 4, 5, 6])
Y = np.array([3000, 3250, 3500, 3750, 4000, 4250])
m = len(lata_pracy)

print(f'Lata pracy: {lata_pracy}')
print(f'Wynagrodzenie: {Y}')
print(f'Liczba próbek: {m}')

Lata pracy: [1 2 3 4 5 6]
Wynagrodzenie: [3000 3250 3500 3750 4000 4250]
Liczba próbek: 6


In [2]:
lata_pracy

array([1, 2, 3, 4, 5, 6])

In [3]:
Y

array([3000, 3250, 3500, 3750, 4000, 4250])

## 3. Macierz cech z kolumną jedynek

Aby wyraz wolny $w_0$ pojawił się w zapisie $X\mathbf{w}$, do macierzy cech dodajemy kolumnę jedynek (tzw. *bias*). Cecha musi być przy tym kolumną o kształcie `(m, 1)`.

In [4]:
X1 = lata_pracy.reshape(m, 1)     # cecha jako kolumna
bias = np.ones((m, 1))            # kolumna jedynek

X = np.hstack([bias, X1])         # połączenie kolumn: [1, x]
print(X)
print(X.shape)

[[1. 1.]
 [1. 2.]
 [1. 3.]
 [1. 4.]
 [1. 5.]
 [1. 6.]]
(6, 2)


## 4. Obliczenia krok po kroku

In [5]:
# 1. X^T X
XtX = X.T @ X
XtX

array([[ 6., 21.],
       [21., 91.]])

In [9]:
# 2. (X^T X)^(-1)
XtX_inv = np.linalg.inv(XtX)
XtX_inv

array([[ 0.86666667, -0.2       ],
       [-0.2       ,  0.05714286]])

In [10]:
# 3. X^T y
Xty = X.T @ Y
Xty

array([21750., 80500.])

In [11]:
# 4. parametry modelu
w = XtX_inv @ Xty
w

array([2750.,  250.])

In [12]:
print(f'Y = {w[0]:.0f} + {w[1]:.0f} * X')

Y = 2750 + 250 * X


In [14]:
lata_pracy

array([1, 2, 3, 4, 5, 6])

In [15]:
Y

array([3000, 3250, 3500, 3750, 4000, 4250])

**Interpretacja:**
- wyraz wolny `2750` – bazowy poziom wynagrodzenia (dla $x = 0$),
- współczynnik `250` – każdy kolejny rok pracy zwiększa wynagrodzenie średnio o 250.

## 5. Porównanie z `LinearRegression`

In [16]:
from sklearn.linear_model import LinearRegression

regression = LinearRegression()
regression.fit(X1, Y)

print(f'intercept_ = {regression.intercept_}')
print(f'coef_      = {regression.coef_[0]}')

intercept_ = 2750.0
coef_      = 250.0


Wynik jest taki sam. `LinearRegression` nie odwraca jednak macierzy „wprost”, tylko używa stabilniejszych numerycznie metod (m.in. rozkładu SVD), więc działa pewniej także dla trudnych danych.

## 6. Podsumowanie

1. Równanie normalne liczy parametry regresji liniowej bez iteracji.
2. Wymaga dodania kolumny jedynek i odwrócenia macierzy $X^T X$.
3. Dla wielu cech jest wolne, a przy współliniowych cechach zawodzi.
4. W praktyce używamy `LinearRegression`, które daje ten sam wynik, ale jest stabilniejsze.